# SNLP Project: LLM Reasoning for Machine Translation
## MVA 2026 — Zebaze et al. (2025) Replication + Extensions

**Authors:** Tom Boumba, Allison Zhuang, Habibi Ahmed Salem, Ruben Cardoso

### Runtime guide:
- **Exp 1 (Thinking vs Non-Thinking):** ~45 min on T4 with Qwen3-0.6B
- **Exp 2 (ICL Few-shot):** ~30 min on T4
- **Exp 3 (CoT Prompting):** ~45 min on T4
- **Exp 4 (Fine-tuning):** ~3 hours on A100 (switch runtime!)

⚠️ **For Exp 4, go to Runtime > Change runtime type > A100**

In [1]:
# ── CELL 1: INSTALL DEPENDENCIES ─────────────────────────────────────────
!pip install -q transformers accelerate datasets sacrebleu evaluate \
    sentencepiece protobuf bitsandbytes peft trl

import torch
print(f'GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU — switch runtime!"}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB' if torch.cuda.is_available() else '')

GPU: Tesla T4
VRAM: 15.6 GB


In [2]:
# ── CELL 2: GLOBAL CONFIG ────────────────────────────────────────────────
# Change these to control what runs

MODEL_NAME = 'Qwen/Qwen3-0.6B'   # Fits on T4 (15GB VRAM)
# MODEL_NAME = 'Qwen/Qwen3-1.7B' # Better quality, needs A100

LANG_PAIRS = [
    ('eng_Latn', 'fra_Latn', 'English', 'French'),   # High resource
    ('fra_Latn', 'eng_Latn', 'French', 'English'),   # Reverse
    ('eng_Latn', 'xho_Latn', 'English', 'Xhosa'),    # Low resource
]

N_EVAL = 50   # Number of test sentences. Increase to 200 if time allows.

import json, re, random, pandas as pd, numpy as np
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
import evaluate
from tqdm.notebook import tqdm

random.seed(42)
np.random.seed(42)

In [3]:
# # ── CELL 3: LOAD MODEL ───────────────────────────────────────────────────
# import torch

# print(f'Loading {MODEL_NAME}...')
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
# model = AutoModelForCausalLM.from_pretrained(
#     MODEL_NAME,
#     torch_dtype=torch.float16,
#     device_map='auto'
# )
# model.eval()
# print('Model loaded!')

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [4]:
from huggingface_hub import login
login()  # or: login(token=os.environ["HF_TOKEN"])

In [5]:
# ── CELL 4: UTILITY FUNCTIONS (batched / parallelized generation) ─────────────

def load_flores(src_lang, tgt_lang, split='devtest', n=50):
    src_ds = load_dataset('openlanguagedata/flores_plus', src_lang, split=split)
    tgt_ds = load_dataset('openlanguagedata/flores_plus', tgt_lang, split=split)
    return [ex['text'] for ex in src_ds][:n], [ex['text'] for ex in tgt_ds][:n]

# Quick test
test_src, test_ref = load_flores('eng_Latn', 'fra_Latn', split='devtest', n=3)
for s, r in zip(test_src, test_ref):
    print(f"EN: {s}\nFR: {r}\n")

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'left'  # important for batched generation with decoder-only LMs

def make_translation_prompt(
    src_text: str,
    src_name: str,
    tgt_name: str,
    model_family: str = "generic",
    thinking: bool = True,
) -> str:
    prompt = (
        f"Please write a high-quality {tgt_name} translation of the following {src_name} sentence\n"
        f"\"{src_text}\"\n"
        "Please provide only the translation, nothing more.\n"
    )

    # Paper's Qwen non-thinking mode
    if model_family.lower().startswith("qwen") and not thinking:
        prompt += "<think>\n\n</think>\n"

    return prompt


import re

def _extract_translation(decoded_text: str) -> str:
    if not decoded_text:
        return ""

    text = decoded_text.strip()

    # Remove thinking traces
    text = re.sub(r"<think>.*?</think>", "", text, flags=re.DOTALL | re.IGNORECASE).strip()

    # Prefer content after "Final Translation" if present
    m = re.search(r"Final Translation\s*:?\s*(.*)$", text, flags=re.DOTALL | re.IGNORECASE)
    if m:
        text = m.group(1).strip()

    # Drop any remaining tags
    text = re.sub(r"</?[^>]+>", "", text).strip()

    # Collapse whitespace
    text = re.sub(r"\s+", " ", text).strip()

    return text.strip(" \"'")


def _build_chat_prompts(sources, prompt_fn, enable_thinking=False):
    prompts = []
    for src in sources:
        msgs = [{'role': 'user', 'content': prompt_fn(src)}]
        prompts.append(
            tokenizer.apply_chat_template(
                msgs,
                tokenize=False,
                add_generation_prompt=True,
                enable_thinking=enable_thinking,
            )
        )
    return prompts


def translate(
    model,
    tokenizer,
    sources,
    prompt_fn,
    max_new_tokens=256,
    enable_thinking=False,
    batch_size=1024,
):
    """
    Batched generation for much better GPU utilization on Colab.
    Falls back to a smaller batch automatically if CUDA OOM happens.
    """
    prompts = _build_chat_prompts(sources, prompt_fn, enable_thinking=enable_thinking)
    translations = []
    i = 0

    pbar = tqdm(total=len(prompts))
    while i < len(prompts):
        current_bs = min(batch_size, len(prompts) - i)

        while current_bs >= 1:
            batch_prompts = prompts[i:i + current_bs]
            try:
                inputs = tokenizer(
                    batch_prompts,
                    return_tensors='pt',
                    padding=True,
                    truncation=True,
                ).to(model.device)

                with torch.no_grad():
                    outputs = model.generate(
                        **inputs,
                        max_new_tokens=max_new_tokens,
                        do_sample=False,
                        temperature=None,
                        top_p=None,
                        pad_token_id=tokenizer.pad_token_id,
                        eos_token_id=tokenizer.eos_token_id,
                        use_cache=True,
                    )

                # IMPORTANT: generated tokens start after the padded input width,
                # not after each sample's non-pad length.
                prompt_width = inputs['input_ids'].shape[1]

                for row in outputs:
                    generated_tokens = row[prompt_width:]
                    decoded = tokenizer.decode(generated_tokens, skip_special_tokens=True)
                    translations.append(_extract_translation(decoded))

                i += current_bs
                pbar.update(current_bs)
                break

            except torch.cuda.OutOfMemoryError:
                torch.cuda.empty_cache()
                current_bs //= 2
                if current_bs == 0:
                    raise RuntimeError(
                        'Out of memory even with batch_size=1. '
                        'Lower max_new_tokens or use a smaller model.'
                    )

    pbar.close()
    return translations


def score(hyps, refs):
    bleu = evaluate.load('sacrebleu')
    chrf = evaluate.load('chrf')
    b = bleu.compute(predictions=hyps, references=[[r] for r in refs])['score']
    c = chrf.compute(predictions=hyps, references=[[r] for r in refs], word_order=2)['score']
    return round(b, 2), round(c, 2)

print('Utilities loaded! Batched translation is enabled.')

Resolving data files:   0%|          | 0/225 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/220 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/225 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/220 [00:00<?, ?it/s]

EN: "We now have 4-month-old mice that are non-diabetic that used to be diabetic," he added.
FR: « Nous avons à présent des souris de 4 mois qui ne sont pas diabétiques alors qu'elles l'étaient auparavant », a-t-il ajouté.

EN: Dr. Ehud Ur, professor of medicine at Dalhousie University in Halifax, Nova Scotia and chair of the clinical and scientific division of the Canadian Diabetes Association cautioned that the research is still in its early days.
FR: Le Dr Ehud Ur, professeur de médecine à l'Université Dalhousie de Halifax (Nouvelle-Écosse) et président de la division clinique et scientifique de l'Association canadienne du diabète, a averti que la recherche en était encore à ses débuts.

EN: Like some other experts, he is skeptical about whether diabetes can be cured, noting that these findings have no relevance to people who already have Type 1 diabetes.
FR: À l'instar d'autres experts, il se montre sceptique quant à la possibilité de guérir le diabète, faisant remarquer que ces ré

In [6]:
# # Quick test
# test_src, test_ref = load_flores('eng_Latn', 'fra_Latn', split='devtest', n=11)
# for s, r in zip(test_src, test_ref):
#     print(f"EN: {s}\nFR: {r}\n\n")

In [7]:
# ── DEBUG FUNCTIONS (batched / parallelized generation) ─────────────

import re
from math import ceil

def contains_think_block(text: str) -> bool:
    if not text:
        return False
    return re.search(r"<think>.*?</think>", text, flags=re.DOTALL | re.IGNORECASE) is not None


def extract_think_block(text: str) -> str | None:
    if not text:
        return None
    m = re.search(r"<think>(.*?)</think>", text, flags=re.DOTALL | re.IGNORECASE)
    if m:
        return m.group(1).strip()
    return None


def translate_with_raw_outputs(
    model,
    tokenizer,
    sources,
    prompt_fn,
    max_new_tokens=512,
    enable_thinking=True,
    batch_size=32,
):
    """
    Returns:
        raw_outputs: decoded model generations before translation extraction
        hyps: extracted translations after _extract_translation(...)
    """
    prompts = _build_chat_prompts(
        sources,
        prompt_fn,
        enable_thinking=enable_thinking,
    )

    raw_outputs = []
    hyps = []
    i = 0

    pbar = tqdm(total=len(prompts))
    while i < len(prompts):
        current_bs = min(batch_size, len(prompts) - i)

        while current_bs >= 1:
            batch_prompts = prompts[i:i + current_bs]

            try:
                inputs = tokenizer(
                    batch_prompts,
                    return_tensors="pt",
                    padding=True,
                    truncation=True,
                ).to(model.device)

                with torch.no_grad():
                    outputs = model.generate(
                        **inputs,
                        max_new_tokens=max_new_tokens,
                        do_sample=False,
                        temperature=None,
                        top_p=None,
                        pad_token_id=tokenizer.pad_token_id,
                        eos_token_id=tokenizer.eos_token_id,
                        use_cache=True,
                    )

                prompt_width = inputs["input_ids"].shape[1]

                for row in outputs:
                    generated_tokens = row[prompt_width:]
                    raw_text = tokenizer.decode(
                        generated_tokens,
                        skip_special_tokens=True
                    ).strip()
                    raw_outputs.append(raw_text)
                    hyps.append(_extract_translation(raw_text))

                i += current_bs
                pbar.update(current_bs)
                break

            except torch.cuda.OutOfMemoryError:
                torch.cuda.empty_cache()
                current_bs //= 2
                if current_bs < 1:
                    raise

    pbar.close()
    return raw_outputs, hyps


def print_raw_output_examples(
    sources,
    references,
    raw_outputs,
    hyps,
    mode_label,
    n_examples=3,
    max_chars=2500,
):
    print(f"\n{'='*100}")
    print(f"RAW OUTPUT EXAMPLES — {mode_label}")
    print(f"{'='*100}")

    for i, (src, ref, raw, hyp) in enumerate(zip(sources[:n_examples], references[:n_examples], raw_outputs[:n_examples], hyps[:n_examples])):
        has_think = contains_think_block(raw)
        think_text = extract_think_block(raw)

        print(f"\n--- Example {i+1} [{mode_label}] ---")
        print(f"SOURCE:\n{src}\n")
        print(f"REFERENCE:\n{ref}\n")
        print(f"HAS <think> BLOCK: {has_think}\n")

        raw_to_show = raw[:max_chars]
        if len(raw) > max_chars:
            raw_to_show += "\n...[truncated]..."

        print(f"RAW MODEL OUTPUT:\n{raw_to_show}\n")

        if think_text:
            think_to_show = think_text[:max_chars]
            if len(think_text) > max_chars:
                think_to_show += "\n...[truncated]..."
            print(f"EXTRACTED THINKING:\n{think_to_show}\n")

        print(f"EXTRACTED TRANSLATION:\n{hyp}\n")

## Experiment 1: Thinking vs Non-Thinking Mode

In [8]:
# # ── EXP 1: THINKING vs NON-THINKING + RAW OUTPUT DEBUG ──────────────────

# N_EVAL = 100   # Number of test sentences. Increase to 200 if time allows.

# exp1_results = []
# exp1_examples = []

# N_EXAMPLES_TO_STORE = 3
# N_EXAMPLES_TO_PRINT = 2
# N_RAW_DEBUG_PRINT = 2

# for src_lang, tgt_lang, src_name, tgt_name in LANG_PAIRS:
#     print(f'\n{"="*100}')
#     print(f'{src_name} → {tgt_name}')
#     print(f'{"="*100}')

#     sources, references = load_flores(src_lang, tgt_lang, n=N_EVAL)

#     for thinking, label in [(True, 'Thinking'), (False, 'Non-Thinking')]:
#         fn = lambda s, sn=src_name, tn=tgt_name, t=thinking: make_translation_prompt(
#             s, sn, tn, "qwen", t
#         )

#         max_new_tokens = 3500
#         if 'Xhosa' in src_name or 'Xhosa' in tgt_name:
#           max_new_tokens = 1024

#         raw_outputs, hyps = translate_with_raw_outputs(
#             model,
#             tokenizer,
#             sources,
#             fn,
#             max_new_tokens=max_new_tokens,
#             enable_thinking=thinking,
#             batch_size=1024,
#         )

#         b, c = score(hyps, references)

#         think_flags = [contains_think_block(x) for x in raw_outputs]
#         think_count = sum(think_flags)
#         think_ratio = think_count / len(raw_outputs) if raw_outputs else 0.0

#         print(
#             f'\n  {label:15s} → BLEU: {b:.2f} | chrF++: {c:.2f} '
#             f'| with <think>: {think_count}/{len(raw_outputs)} ({100*think_ratio:.1f}%)'
#         )

#         # Print a few raw outputs so you can inspect reasoning / extra text
#         print_raw_output_examples(
#             sources=sources,
#             references=references,
#             raw_outputs=raw_outputs,
#             hyps=hyps,
#             mode_label=f"{src_name} → {tgt_name} | {label}",
#             n_examples=N_RAW_DEBUG_PRINT,
#         )

#         examples = []
#         for i, (src, ref, raw, hyp, has_think) in enumerate(
#             zip(
#                 sources[:N_EXAMPLES_TO_STORE],
#                 references[:N_EXAMPLES_TO_STORE],
#                 raw_outputs[:N_EXAMPLES_TO_STORE],
#                 hyps[:N_EXAMPLES_TO_STORE],
#                 think_flags[:N_EXAMPLES_TO_STORE],
#             )
#         ):
#             ex = {
#                 'example_id': i,
#                 'source': src,
#                 'reference': ref,
#                 'raw_output': raw,
#                 'contains_think_block': has_think,
#                 'think_block': extract_think_block(raw),
#                 'hypothesis': hyp,
#             }
#             examples.append(ex)

#             exp1_examples.append({
#                 'src': src_name,
#                 'tgt': tgt_name,
#                 'mode': label,
#                 'example_id': i,
#                 'source': src,
#                 'reference': ref,
#                 'raw_output': raw,
#                 'contains_think_block': has_think,
#                 'think_block': extract_think_block(raw),
#                 'hypothesis': hyp,
#             })

#         exp1_results.append({
#             'src': src_name,
#             'tgt': tgt_name,
#             'mode': label,
#             'bleu': b,
#             'chrf': c,
#             'n_outputs': len(raw_outputs),
#             'n_with_think_block': think_count,
#             'prop_with_think_block': think_ratio,
#             'examples': examples,
#         })

# # Summary table
# df1 = pd.DataFrame([
#     {k: v for k, v in r.items() if k != 'examples'}
#     for r in exp1_results
# ])

# df1_examples = pd.DataFrame(exp1_examples)

# print('\n' + '='*100)
# print('SUMMARY RESULTS')
# print('='*100)
# print(df1.to_string(index=False))

# # Optional aggregated summary across directions
# df1_think_summary = (
#     df1.groupby('mode', as_index=False)
#       .agg({
#           'n_outputs': 'sum',
#           'n_with_think_block': 'sum',
#           'bleu': 'mean',
#           'chrf': 'mean',
#       })
# )

# df1_think_summary['prop_with_think_block'] = (
#     df1_think_summary['n_with_think_block'] / df1_think_summary['n_outputs']
# )

# print('\n' + '='*100)
# print('THINK BLOCK SUMMARY BY MODE')
# print('='*100)
# print(df1_think_summary.to_string(index=False))

# # Save everything
# with open('results_exp1.json', 'w', encoding='utf-8') as f:
#     json.dump(exp1_results, f, indent=2, ensure_ascii=False)

# df1.to_csv('results_exp1_summary.csv', index=False, encoding='utf-8')
# df1_examples.to_csv('results_exp1_examples.csv', index=False, encoding='utf-8')
# df1_think_summary.to_csv('results_exp1_think_block_summary.csv', index=False, encoding='utf-8')

## Experiment 2: Few-shot MT

In [9]:
# # ── EXP 2: FEW-SHOT MT WITH GEMMA-1B, FAIR NESTED ICL SETUP ─────────────

# import gc
# import json
# import random
# import numpy as np
# import pandas as pd
# import torch
# from transformers import AutoTokenizer, AutoModelForCausalLM

# # -----------------------------
# # Config
# # -----------------------------
# GEMMA_MODEL_ID = "google/gemma-3-1b-it"
# ICL_K_VALUES = [0, 1, 2, 4, 8]   # nested prefixes: fair comparison across k
# N_EVAL_EXP2 = 50
# SEED_EXP2 = 13
# DTYPE = torch.bfloat16 if torch.cuda.is_available() else torch.float32

# # We need at least max(k) support examples plus evaluation examples.
# K_MAX = max(ICL_K_VALUES)
# N_TOTAL_NEEDED = K_MAX + N_EVAL_EXP2

# random.seed(SEED_EXP2)
# np.random.seed(SEED_EXP2)
# torch.manual_seed(SEED_EXP2)

# # # -----------------------------
# # # Optional: free previous model
# # # -----------------------------
# # try:
# #     del model
# # except NameError:
# #     pass

# # try:
# #     del tokenizer
# # except NameError:
# #     pass

# # gc.collect()
# # if torch.cuda.is_available():
# #     torch.cuda.empty_cache()

# # -----------------------------
# # Load Gemma-1B IT
# # -----------------------------
# tokenizer = AutoTokenizer.from_pretrained(GEMMA_MODEL_ID)
# model = AutoModelForCausalLM.from_pretrained(
#     GEMMA_MODEL_ID,
#     torch_dtype=DTYPE,
#     device_map="auto",
# )
# model.eval()

# # -----------------------------
# # Helpers
# # -----------------------------
# def make_fewshot_translation_prompt(src_text, src_name, tgt_name, demonstrations):
#     """
#     Few-shot prompt for MT.
#     demonstrations: list of (src_sent, tgt_sent)
#     """
#     lines = [
#         f"Translate from {src_name} to {tgt_name}.",
#         "Use the examples below to infer the translation pattern.",
#         "Return only the translation, nothing else.",
#         "",
#     ]

#     if demonstrations:
#         lines.append("Examples:")
#         for i, (ex_src, ex_tgt) in enumerate(demonstrations, start=1):
#             lines.extend([
#                 f"Example {i}",
#                 f"{src_name}: {ex_src}",
#                 f"{tgt_name}: {ex_tgt}",
#                 ""
#             ])

#     lines.extend([
#         "Now translate:",
#         f"{src_name}: {src_text}",
#         f"{tgt_name}:"
#     ])
#     return "\n".join(lines)


# def get_fair_support_and_eval(src_lang, tgt_lang, n_eval, k_max):
#     """
#     Fair setup:
#     - fetch exactly k_max + n_eval aligned examples
#     - use the FIRST k_max as the fixed support pool
#     - use the REMAINDER as the eval set
#     This guarantees:
#       * same eval set for all k
#       * same support pool for all k
#       * k-shot prompts use nested prefixes of the same pool
#     """
#     sources_all, references_all = load_flores(src_lang, tgt_lang, n=k_max + n_eval)

#     support_sources = sources_all[:k_max]
#     support_refs = references_all[:k_max]

#     eval_sources = sources_all[k_max:k_max + n_eval]
#     eval_refs = references_all[k_max:k_max + n_eval]

#     support_pairs = list(zip(support_sources, support_refs))
#     return support_pairs, eval_sources, eval_refs


# def summarize_prompt_length_chars(prompt_fn, sources, n_probe=5):
#     probe = sources[:min(n_probe, len(sources))]
#     if not probe:
#         return 0.0
#     return float(np.mean([len(prompt_fn(x)) for x in probe]))


# # -----------------------------
# # Run experiment
# # -----------------------------
# exp2_results = []
# exp2_support_sets = []

# for src_lang, tgt_lang, src_name, tgt_name in LANG_PAIRS:
#     print(f"\n{'='*100}")
#     print(f"GEMMA FEW-SHOT ICL — {src_name} → {tgt_name}")
#     print(f"{'='*100}")

#     support_pairs, eval_sources, eval_refs = get_fair_support_and_eval(
#         src_lang=src_lang,
#         tgt_lang=tgt_lang,
#         n_eval=N_EVAL_EXP2,
#         k_max=K_MAX,
#     )

#     # Save the fixed support pool used for this direction
#     exp2_support_sets.append({
#         "src": src_name,
#         "tgt": tgt_name,
#         "k_max": K_MAX,
#         "support_examples": [
#             {
#                 "example_id": i,
#                 "source": s,
#                 "target": t,
#             }
#             for i, (s, t) in enumerate(support_pairs)
#         ]
#     })

#     # Show the support pool once so fairness is explicit
#     print(f"Fixed support pool size: {len(support_pairs)}")
#     for i, (s, t) in enumerate(support_pairs[:min(2, len(support_pairs))], start=1):
#         print(f"\nSupport example {i}")
#         print(f"{src_name}: {s}")
#         print(f"{tgt_name}: {t}")

#     for k in ICL_K_VALUES:
#         demos_k = support_pairs[:k]   # nested prefix => fair comparison

#         fn = lambda s, sn=src_name, tn=tgt_name, demos=demos_k: make_fewshot_translation_prompt(
#             s, sn, tn, demos
#         )

#         avg_prompt_chars = summarize_prompt_length_chars(fn, eval_sources, n_probe=5)

#         hyps = translate(
#             model,
#             tokenizer,
#             eval_sources,
#             fn,
#             max_new_tokens=160,
#             enable_thinking=False,   # not relevant for Gemma here
#             batch_size=16,
#         )

#         bleu, chrf = score(hyps, eval_refs)

#         print(
#             f"k={k:2d}  "
#             f"BLEU: {bleu:6.2f}  "
#             f"chrF++: {chrf:6.2f}  "
#             f"avg prompt chars: {avg_prompt_chars:7.1f}"
#         )

#         exp2_results.append({
#             "src": src_name,
#             "tgt": tgt_name,
#             "k": k,
#             "bleu": bleu,
#             "chrf": chrf,
#             "n_eval": len(eval_sources),
#             "avg_prompt_chars": avg_prompt_chars,
#             "translations": hyps[:3],   # quick qualitative inspection
#         })

# # -----------------------------
# # Summaries and saving
# # -----------------------------
# df2 = pd.DataFrame([
#     {k: v for k, v in row.items() if k != "translations"}
#     for row in exp2_results
# ])

# print(f"\n{'='*100}")
# print("EXP 2 SUMMARY")
# print(f"{'='*100}")
# print(df2.to_string(index=False))

# # Mean over directions for each k
# df2_mean = (
#     df2.groupby("k", as_index=False)
#        .agg({
#            "bleu": "mean",
#            "chrf": "mean",
#            "avg_prompt_chars": "mean",
#        })
#        .sort_values("k")
# )

# print(f"\n{'='*100}")
# print("MEAN ACROSS DIRECTIONS")
# print(f"{'='*100}")
# print(df2_mean.to_string(index=False))

# with open("results_exp2_gemma_fewshot.json", "w", encoding="utf-8") as f:
#     json.dump(exp2_results, f, indent=2, ensure_ascii=False)

# with open("results_exp2_gemma_support_sets.json", "w", encoding="utf-8") as f:
#     json.dump(exp2_support_sets, f, indent=2, ensure_ascii=False)

# df2.to_csv("results_exp2_gemma_fewshot.csv", index=False, encoding="utf-8")
# df2_mean.to_csv("results_exp2_gemma_fewshot_mean_by_k.csv", index=False, encoding="utf-8")

In [10]:
# ── EXP 3: ZERO-SHOT CoT PROMPTING FOR MT WITH GEMMA-1B ─────────────────

import re
import json
import pandas as pd
from tqdm.notebook import tqdm

# -------------------------------------------------------------------------
# Exact CoT construction templates from Appendix A.3 of the paper
# -------------------------------------------------------------------------

COT_TEMPLATES = {
    "T1": """<think>
1. Analyze the sentence structure and identify the core elements (subject, verb, object).
2. Translate the sentence from the origin language to the target language, focusing on the core elements.
3. Review the translation for basic accuracy and grammatical structure.
4. Identify areas that need further refinement (e.g., word choice, tense, or word order).
5. Modify the translation to improve fluency and coherence, considering the context.
6. Finalize the translation by ensuring it retains the original meaning while improving readability.
</think>""",

    "T2": """<think>
1. Identify basic elements: Break down the sentence into its main components and identify the key subject, verb, and object.
2. Translate to intermediate language: Convert these elements into an intermediate language structure (e.g., simple syntactic rules or function names).
3. Refine back to target language: Translate from the intermediate language back to the target language, adjusting for syntactic norms and idiomatic expressions.
4. Check for accuracy: Ensure that the meaning is preserved in the translated sentence by checking noun-verb agreement and connectors.
5. Adjust word order: Modify word order to ensure that it aligns with the target language’s grammatical structure.
6. Final refinement: Review the translation for naturalness, idiomatic use, and overall flow.
</think>""",

    "T3": """<think>
1. Analyze the provided context in the source language.
2. Translate the source text to the target language.
3. Perform back translation from the target language to the source language.
4. Compare the back translation with the original source context.
5. Evaluate whether the meaning of the back translation aligns with the original.
6. If discrepancies are identified, adjust the target language translation to enhance consistency with the original meaning.
7. Finalize the translation by ensuring both forward and back translations accurately align across all languages involved.
</think>""",

    "T4": """<think>
1. Analyze the current sentence, along with the previous sentences, to understand the overall conversation context.
2. Identify key elements like tone, formality, or subject matter based on the ongoing conversation.
3. Translate the sentence while ensuring that the translation is aligned with the tone, style, and subject of the preceding dialogue.
4. If any ambiguity exists in the translation due to context, refine the translation to better fit the conversation flow.
5. Verify that the translation maintains coherence with the larger conversation, ensuring consistency in language and tone.
6. Finalize the translation by cross-checking it with the conversation’s context to ensure it feels natural and appropriately aligned.
</think>""",

    "T5": """<think>
1. Analyze the source sentence and identify the key elements (verbs, subjects, objects, etc.).
2. Based on these elements, determine the most suitable translation strategy (literal vs. idiomatic).
3. Select the best translation for each word or phrase, considering context and language-specific structures.
4. Explain the rationale behind choosing specific words or phrases.
5. After completing the initial translation, review each translation decision and explain any adjustments made for fluency or accuracy.
6. Provide a final explanation for the translation choices, discussing any trade-offs made between literal meaning and contextual appropriateness.
</think>""",

    "T6": """<think>
1. Analyze the sentence’s syntactic structure in the source language (e.g., identify whether it’s active or passive).
2. Determine the most appropriate syntactic structure in the target language (e.g., whether it needs to be rephrased from active to passive or vice versa).
3. Adjust the word order and grammatical structure in the target language to match the sentence’s meaning, while maintaining clarity.
4. Translate the sentence, ensuring that subject-verb-object relationships and other syntactic elements align with target language norms.
5. After the translation, check the sentence’s grammar and overall flow in the target language, making sure it is clear and fluid.
6. If the sentence feels awkward or unnatural, refine the structure by adjusting word choice or reordering components.
</think>""",
}

STRATEGY_LABELS = {
    "cot_zero_shot": "Let's think step by step",
    "T1": "T1 - Hierarchical Translation",
    "T2": "T2 - Triangulating Translation",
    "T3": "T3 - Back Translation",
    "T4": "T4 - Context-aware Translation",
    "T5": "T5 - Translation Explanation",
    "T6": "T6 - Structural Transformation",
}

# -------------------------------------------------------------------------
# Prompt builder
# -------------------------------------------------------------------------

def make_cot_translation_prompt(
    src_text: str,
    src_name: str,
    tgt_name: str,
    strategy: str = "cot_zero_shot",
) -> str:
    """
    Zero-shot CoT prompting for MT with explicit <think>...</think> markup
    plus 'Final Translation' marker for easy extraction.
    """

    base = (
        f"Please write a high-quality {tgt_name} translation of the following {src_name} sentence.\n\n"
        f"{src_name}: {src_text}\n\n"
    )

    if strategy == "cot_zero_shot":
        reasoning_instruction = (
            "Let's think step by step.\n"
            "First reason about the translation inside <think> and </think>.\n"
            "Then output the final translation after the line 'Final Translation'.\n"
            "Do not put anything except reasoning inside <think>...</think>.\n"
            "Do not put anything after the final translation.\n"
        )
    elif strategy in COT_TEMPLATES:
        reasoning_instruction = (
            "You should reason before translating.\n"
            "Write your reasoning inside <think> and </think>.\n"
            "Follow this Thinking Chain Guide as closely as possible:\n"
            f"{COT_TEMPLATES[strategy]}\n\n"
            "Then output the final translation after the line 'Final Translation'.\n"
            "Do not put anything except reasoning inside <think>...</think>.\n"
            "Do not put anything after the final translation.\n"
        )
    else:
        raise ValueError(f"Unknown strategy: {strategy}")

    format_hint = (
        "Output format:\n"
        "<think>\n"
        "...\n"
        "</think>\n\n"
        "Final Translation\n"
        "<your translation here>"
    )

    return base + reasoning_instruction + "\n" + format_hint


# -------------------------------------------------------------------------
# Extraction utilities
# -------------------------------------------------------------------------

def _extract_translation_from_cot(decoded_text: str) -> str:
    """
    Extract final translation from outputs of the form:
      <think> ... </think>
      Final Translation
      ...
    Also robust to imperfect outputs.
    """
    if not decoded_text:
        return ""

    text = decoded_text.strip()

    # Prefer content after the last 'Final Translation'
    m = re.search(r"Final Translation\s*:?\s*(.*)$", text, flags=re.DOTALL | re.IGNORECASE)
    if m:
        candidate = m.group(1).strip()
        candidate = re.sub(r"</?[^>]+>", "", candidate).strip()
        candidate = re.sub(r"\s+", " ", candidate).strip()
        return candidate.strip(" \"'")

    # Fallback: remove the think block, then clean
    text_wo_think = re.sub(r"<think>.*?</think>", "", text, flags=re.DOTALL | re.IGNORECASE).strip()
    text_wo_think = re.sub(r"</?[^>]+>", "", text_wo_think).strip()
    text_wo_think = re.sub(r"\s+", " ", text_wo_think).strip()

    return text_wo_think.strip(" \"'")


def _extract_think_block(decoded_text: str) -> str | None:
    if not decoded_text:
        return None
    m = re.search(r"<think>(.*?)</think>", decoded_text, flags=re.DOTALL | re.IGNORECASE)
    return m.group(1).strip() if m else None


def _contains_think_block(decoded_text: str) -> bool:
    if not decoded_text:
        return False
    return re.search(r"<think>.*?</think>", decoded_text, flags=re.DOTALL | re.IGNORECASE) is not None


# -------------------------------------------------------------------------
# Raw-output translation helper
# Assumes you already have:
#   - _build_chat_prompts(...)
#   - model, tokenizer
# -------------------------------------------------------------------------

def translate_with_raw_outputs_cot(
    model,
    tokenizer,
    sources,
    prompt_fn,
    max_new_tokens=512,
    batch_size=16,
):
    prompts = _build_chat_prompts(
        sources,
        prompt_fn,
        enable_thinking=False,  # Gemma HF path: prompt-controlled CoT, not model kwarg
    )

    raw_outputs = []
    hyps = []

    i = 0
    pbar = tqdm(total=len(prompts), desc="Generating", leave=False)
    while i < len(prompts):
        current_bs = min(batch_size, len(prompts) - i)
        batch_prompts = prompts[i:i + current_bs]

        inputs = tokenizer(
            batch_prompts,
            return_tensors="pt",
            padding=True,
            truncation=True,
        ).to(model.device)

        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                do_sample=False,
                temperature=None,
                top_p=None,
                pad_token_id=tokenizer.pad_token_id,
                eos_token_id=tokenizer.eos_token_id,
                use_cache=True,
            )

        prompt_width = inputs["input_ids"].shape[1]

        for row in outputs:
            generated_tokens = row[prompt_width:]
            raw_text = tokenizer.decode(generated_tokens, skip_special_tokens=True).strip()
            raw_outputs.append(raw_text)
            hyps.append(_extract_translation_from_cot(raw_text))

        i += current_bs
        pbar.update(current_bs)
    pbar.close()

    return raw_outputs, hyps


# -------------------------------------------------------------------------
# Experiment runner
# -------------------------------------------------------------------------

COT_STRATEGIES = ["cot_zero_shot", "T1", "T2", "T3", "T4", "T5", "T6"]

def run_exp3_cot_prompting(
    model,
    tokenizer,
    lang_pairs,
    n_eval=50,
    max_new_tokens=512,
    batch_size=16,
    n_examples_to_store=3,
    n_examples_to_print=1,
):
    exp3_results = []

    for src_lang, tgt_lang, src_name, tgt_name in tqdm(lang_pairs, desc="Language pairs"):
        print(f"\n{'='*100}")
        print(f"EXP 3 — CoT prompting with Gemma-1B: {src_name} → {tgt_name}")
        print(f"{'='*100}")

        sources, references = load_flores(src_lang, tgt_lang, n=n_eval)

        for strategy in tqdm(COT_STRATEGIES, desc="Strategies"):
            tqdm.write(f"Running strategy: {STRATEGY_LABELS[strategy]}")

            label = STRATEGY_LABELS[strategy]

            fn = lambda s, sn=src_name, tn=tgt_name, strat=strategy: make_cot_translation_prompt(
                s, sn, tn, strat
            )

            raw_outputs, hyps = translate_with_raw_outputs_cot(
                model=model,
                tokenizer=tokenizer,
                sources=sources,
                prompt_fn=fn,
                max_new_tokens=max_new_tokens,
                batch_size=batch_size,
            )

            bleu, chrf = score(hyps, references)
            think_flags = [_contains_think_block(x) for x in raw_outputs]
            think_ratio = sum(think_flags) / len(think_flags) if think_flags else 0.0

            print(
                f"{label:35s} | BLEU: {bleu:6.2f} | chrF++: {chrf:6.2f} | "
                f"with <think>: {100*think_ratio:5.1f}%"
            )

            examples = []
            for i, (src, ref, raw, hyp, has_think) in enumerate(
                zip(
                    sources[:n_examples_to_store],
                    references[:n_examples_to_store],
                    raw_outputs[:n_examples_to_store],
                    hyps[:n_examples_to_store],
                    think_flags[:n_examples_to_store],
                )
            ):
                examples.append({
                    "example_id": i,
                    "source": src,
                    "reference": ref,
                    "raw_output": raw,
                    "contains_think_block": has_think,
                    "think_block": _extract_think_block(raw),
                    "hypothesis": hyp,
                })

            if n_examples_to_print > 0:
                print(f"  Example output for {label}:")
                for ex in examples[:n_examples_to_print]:
                    print("  ---")
                    print("  SOURCE:", ex["source"])
                    print("  REFERENCE:", ex["reference"])
                    print("  RAW OUTPUT:", ex["raw_output"][:800].replace("\n", " "))
                    print("  EXTRACTED:", ex["hypothesis"])

            exp3_results.append({
                "src": src_name,
                "tgt": tgt_name,
                "strategy": strategy,
                "strategy_label": label,
                "bleu": bleu,
                "chrf": chrf,
                "n_outputs": len(raw_outputs),
                "n_with_think_block": int(sum(think_flags)),
                "prop_with_think_block": think_ratio,
                "examples": examples,
            })

    return exp3_results

In [ ]:
# -------------------------------------------------------------------------
# Suggested call
# -------------------------------------------------------------------------

GEMMA_MODEL_ID = "google/gemma-3-1b-it"
DTYPE = torch.bfloat16 if torch.cuda.is_available() else torch.float32

# -----------------------------
# Load Gemma-1B IT
# -----------------------------
tokenizer = AutoTokenizer.from_pretrained(GEMMA_MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    GEMMA_MODEL_ID,
    torch_dtype=DTYPE,
    device_map="auto",
)
model.eval()

# Example:
exp3_results = run_exp3_cot_prompting(
    model=model,
    tokenizer=tokenizer,
    lang_pairs=LANG_PAIRS,
    n_eval=50,
    max_new_tokens=2048,
    batch_size=1024,
    n_examples_to_store=3,
    n_examples_to_print=1,
)

df3 = pd.DataFrame([
    {k: v for k, v in row.items() if k != "examples"}
    for row in exp3_results
])
print(df3.to_string(index=False))

with open("results_exp3_cot_prompting.json", "w", encoding="utf-8") as f:
    json.dump(exp3_results, f, indent=2, ensure_ascii=False)

df3.to_csv("results_exp3_cot_prompting.csv", index=False, encoding="utf-8")

`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/340 [00:00<?, ?it/s]

Language pairs:   0%|          | 0/3 [00:00<?, ?it/s]


EXP 3 — CoT prompting with Gemma-1B: English → French


Resolving data files:   0%|          | 0/225 [00:00<?, ?it/s]

...

Resolving data files:   0%|          | 0/220 [00:00<?, ?it/s]

Strategies:   0%|          | 0/7 [00:00<?, ?it/s]

Running strategy: Let's think step by step


Generating:   0%|          | 0/50 [00:00<?, ?it/s]

The following generation flags are not valid and may be ignored: ['top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
